# 02 — Transformação e Qualidade dos Dados — Camada Silver

## Objetivo

Este notebook corresponde à etapa de transformação dos dados do pipeline do projeto de análise do mercado imobiliário.

Os dados provenientes da camada Bronze serão submetidos a processos de inspeção, limpeza, padronização e validação de qualidade antes de serem armazenados na camada Silver.

Nesta etapa serão avaliados aspectos como valores ausentes, registros duplicados, consistência dos tipos de dados, padronização de campos categóricos e possíveis valores fora dos padrões esperados.

## Arquitetura

**Camada Bronze → Camada Silver**

**Origem:** `mvp_imobiliario.bronze.mercado_imobiliario_raw`

**Destino:** `mvp_imobiliario.silver.mercado_imobiliario_tratado`

A camada Bronze permanece preservada como registro dos dados ingeridos. As transformações realizadas neste notebook serão aplicadas somente aos dados destinados à camada Silver.

## 1. Leitura dos dados da camada Bronze

Inicialmente, a tabela consolidada na camada Bronze é carregada para um DataFrame Spark.

Nesta etapa nenhuma transformação é realizada. O objetivo é recuperar os dados persistidos anteriormente e confirmar a quantidade de registros disponível antes do início dos procedimentos de qualidade e transformação.

In [0]:
# Leitura da tabela da camada Bronze
tabela_bronze = "mvp_imobiliario.bronze.mercado_imobiliario_raw"

df_bronze = spark.table(tabela_bronze)

print(f"Quantidade de registros na Bronze: {df_bronze.count()}")

display(df_bronze.limit(10))

Quantidade de registros na Bronze: 36933


mes,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio,arquivo_origem,cidade,uf,data_ingestao
2025-09-01,Abranches,venda,5941,4341,6781,42,1,511,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Água Verde,aluguel,50,41,59,4,0,286,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Água Verde,venda,11771,9692,14377,533,11,384,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Ahú,aluguel,48,48,48,2,0,587,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Ahú,venda,12235,11027,13344,126,1,352,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto Boqueirão,aluguel,25,24,25,3,0,144,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto Boqueirão,venda,4866,4375,5513,58,1,366,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Glória,aluguel,72,52,86,14,0,784,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Glória,venda,12762,10920,15346,86,1,443,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Rua XV,aluguel,26,26,26,1,0,104,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z


## 2. Diagnóstico inicial da qualidade dos dados

Antes da aplicação das transformações da camada Silver, é realizado um diagnóstico dos dados provenientes da camada Bronze.

O objetivo é identificar problemas relacionados à completude, consistência e unicidade dos registros, permitindo documentar as condições encontradas na fonte e justificar os tratamentos realizados posteriormente.

Nesta etapa serão avaliados:

1 estrutura e tipos dos atributos;

2 valores nulos;

3 registros duplicados;

4 valores distintos dos principais campos categóricos;

5 consistência dos valores numéricos.

6 Construção da Camada Silver

### 1 estrutura e tipos dos atributos;

In [0]:
# Estrutura dos dados recebidos da camada Bronze
df_bronze.printSchema()

root
 |-- mes: date (nullable = true)
 |-- bairro: string (nullable = true)
 |-- transacao: string (nullable = true)
 |-- mediana_m2: integer (nullable = true)
 |-- p25_m2: integer (nullable = true)
 |-- p75_m2: integer (nullable = true)
 |-- anuncios_ativos_media_dia: integer (nullable = true)
 |-- novos_anuncios: integer (nullable = true)
 |-- dias_no_mercado_medio: integer (nullable = true)
 |-- arquivo_origem: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)



### 2 valores nulos;

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# Diagnóstico de completude: quantidade de valores nulos por coluna
df_nulos = df_bronze.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_bronze.columns
])

display(df_nulos)

mes,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio,arquivo_origem,cidade,uf,data_ingestao
0,0,0,3756,3756,3756,0,0,22,0,0,0,0


### 2.1 Análise de completude

A avaliação de completude identificou valores ausentes em alguns dos atributos do conjunto de dados.

Antes da aplicação de qualquer tratamento, os registros com valores nulos serão analisados considerando sua distribuição entre cidades e tipos de transação. Essa investigação permite compreender a origem das ausências e evita a exclusão ou imputação de dados sem justificativa.

In [0]:
from pyspark.sql.functions import col, count, when

# Investiga a distribuição dos valores nulos das métricas de preço
df_nulos_preco = (
    df_bronze
    .groupBy("cidade", "uf", "transacao")
    .agg(
        count("*").alias("total_registros"),
        count(when(col("mediana_m2").isNull(), 1)).alias("nulos_mediana_m2"),
        count(when(col("p25_m2").isNull(), 1)).alias("nulos_p25_m2"),
        count(when(col("p75_m2").isNull(), 1)).alias("nulos_p75_m2")
    )
    .filter(
        (col("nulos_mediana_m2") > 0) |
        (col("nulos_p25_m2") > 0) |
        (col("nulos_p75_m2") > 0)
    )
    .orderBy("cidade", "transacao")
)

display(df_nulos_preco)

cidade,uf,transacao,total_registros,nulos_mediana_m2,nulos_p25_m2,nulos_p75_m2
curitiba,pr,aluguel,761,35,35,35
curitiba,pr,venda,1036,11,11,11
recife,pe,aluguel,609,35,35,35
recife,pe,venda,986,121,121,121
rio-de-janeiro,rj,aluguel,1043,191,191,191
rio-de-janeiro,rj,venda,1932,111,111,111
salvador,ba,aluguel,1088,81,81,81
salvador,ba,venda,2079,143,143,143
sao-paulo,sp,aluguel,10085,1698,1698,1698
sao-paulo,sp,venda,17314,1330,1330,1330


### 2.2 Consistência dos valores ausentes nas métricas de preço

As três métricas de preço (`mediana_m2`, `p25_m2` e `p75_m2`) apresentaram a mesma quantidade de valores ausentes: 3.756 registros.

Para verificar se essas ausências ocorrem simultaneamente nos mesmos registros, será realizada uma validação das combinações de nulidade entre os três atributos. Essa análise é necessária antes da definição da estratégia de tratamento dos dados.

In [0]:
from pyspark.sql.functions import col, when, count

# Verificação das combinações de nulidade das métricas de preço
df_padrao_nulos = (
    df_bronze
    .select(
        when(col("mediana_m2").isNull(), "nulo").otherwise("preenchido")
            .alias("mediana_m2"),
        when(col("p25_m2").isNull(), "nulo").otherwise("preenchido")
            .alias("p25_m2"),
        when(col("p75_m2").isNull(), "nulo").otherwise("preenchido")
            .alias("p75_m2")
    )
    .groupBy("mediana_m2", "p25_m2", "p75_m2")
    .agg(count("*").alias("quantidade"))
    .orderBy("mediana_m2", "p25_m2", "p75_m2")
)

display(df_padrao_nulos)

mediana_m2,p25_m2,p75_m2,quantidade
nulo,nulo,nulo,3756
preenchido,preenchido,preenchido,33177


### 2.2.1 Conclusão da análise de valores ausentes

A análise demonstrou que os valores ausentes das métricas `mediana_m2`, `p25_m2` e `p75_m2` ocorrem de forma conjunta.

Dos **36.933 registros** da camada Bronze:

- **33.177 registros** possuem as três métricas de preço preenchidas;
- **3.756 registros** apresentam simultaneamente valores nulos nas três métricas.

Não foram identificados registros com preenchimento parcial entre essas três variáveis. Esse comportamento indica que a ausência das informações de preço ocorre de forma estruturada na fonte de dados, e não de maneira isolada entre os atributos.

Os valores ausentes não serão substituídos por zero, pois a ausência de informação de preço não representa preço igual a zero. O tratamento desses registros será definido de acordo com sua utilização nas análises posteriores, evitando a introdução de valores artificiais que possam distorcer os resultados.

> ## 3 registros duplicados;

### Verificação de registros duplicados

Nesta etapa será verificada a existência de registros duplicados na camada Bronze.

Como cada registro representa uma observação mensal do mercado imobiliário para determinada combinação de cidade, bairro e tipo de transação, duplicidades podem provocar contagens indevidas e distorções nas análises posteriores.

Primeiramente será verificada a existência de linhas completamente duplicadas. Em seguida, caso sejam encontradas duplicidades, os registros serão analisados antes da aplicação de qualquer tratamento.

In [0]:
from pyspark.sql.functions import count

# Quantidade total de registros
total_registros = df_bronze.count()

# Registros distintos considerando todas as colunas
total_distintos = df_bronze.distinct().count()

# Quantidade de registros duplicados
total_duplicados = total_registros - total_distintos

print(f"Total de registros: {total_registros}")
print(f"Registros distintos: {total_distintos}")
print(f"Registros duplicados: {total_duplicados}")

Total de registros: 36933
Registros distintos: 36933
Registros duplicados: 0


### 3.1 Conclusão da análise de duplicidade

A verificação de duplicidade identificou **36.933 registros totais e 36.933 registros distintos**, resultando em **0 registros completamente duplicados**.

Dessa forma, não foi necessária a aplicação de procedimentos para remoção de duplicidades nesta etapa. O resultado também demonstra que a consolidação das cinco fontes de dados realizada na camada Bronze não introduziu linhas integralmente repetidas no conjunto de dados.

## 4 valores distintos dos principais campos categóricos;

### Análise dos campos categóricos

Nesta etapa são avaliados os valores distintos dos principais atributos categóricos do conjunto de dados, com o objetivo de identificar possíveis inconsistências de preenchimento, diferenças de grafia, uso inadequado de maiúsculas e minúsculas, espaços adicionais ou categorias inesperadas.

Serão avaliados os campos `cidade`, `uf` e `transacao`, que possuem importância direta na segmentação e comparação dos dados do mercado imobiliário.

In [0]:
# Valores distintos dos principais campos categóricos

print("CIDADES:")
df_bronze.select("cidade").distinct().orderBy("cidade").show(100, truncate=False)

print("UNIDADES FEDERATIVAS:")
df_bronze.select("uf").distinct().orderBy("uf").show(100, truncate=False)

print("TIPOS DE TRANSAÇÃO:")
df_bronze.select("transacao").distinct().orderBy("transacao").show(100, truncate=False)

CIDADES:
+--------------+
|cidade        |
+--------------+
|curitiba      |
|recife        |
|rio-de-janeiro|
|salvador      |
|sao-paulo     |
+--------------+

UNIDADES FEDERATIVAS:
+---+
|uf |
+---+
|ba |
|pe |
|pr |
|rj |
|sp |
+---+

TIPOS DE TRANSAÇÃO:
+---------+
|transacao|
+---------+
|aluguel  |
|venda    |
+---------+



In [0]:
from pyspark.sql.functions import col, trim, lower, count

# Verificação de possíveis inconsistências nos campos categóricos
for campo in ["cidade", "uf", "transacao"]:
    print(f"\nCAMPO: {campo.upper()}")

    (
        df_bronze
        .groupBy(
            col(campo),
            lower(trim(col(campo))).alias("valor_padronizado")
        )
        .agg(count("*").alias("quantidade"))
        .orderBy("valor_padronizado", col(campo))
        .show(200, truncate=False)
    )


CAMPO: CIDADE
+--------------+-----------------+----------+
|cidade        |valor_padronizado|quantidade|
+--------------+-----------------+----------+
|curitiba      |curitiba         |1797      |
|recife        |recife           |1595      |
|rio-de-janeiro|rio-de-janeiro   |2975      |
|salvador      |salvador         |3167      |
|sao-paulo     |sao-paulo        |27399     |
+--------------+-----------------+----------+


CAMPO: UF
+---+-----------------+----------+
|uf |valor_padronizado|quantidade|
+---+-----------------+----------+
|ba |ba               |3167      |
|pe |pe               |1595      |
|pr |pr               |1797      |
|rj |rj               |2975      |
|sp |sp               |27399     |
+---+-----------------+----------+


CAMPO: TRANSACAO
+---------+-----------------+----------+
|transacao|valor_padronizado|quantidade|
+---------+-----------------+----------+
|aluguel  |aluguel          |13586     |
|venda    |venda            |23347     |
+---------+---------

### 4.1 Conclusão da análise dos campos categóricos

A análise dos atributos categóricos `cidade`, `uf` e `transacao` não identificou inconsistências de preenchimento.

Foram encontradas cinco cidades (`curitiba`, `recife`, `rio-de-janeiro`, `salvador` e `sao-paulo`), cinco Unidades Federativas correspondentes (`pr`, `pe`, `rj`, `ba` e `sp`) e dois tipos de transação (`aluguel` e `venda`).

A comparação entre os valores originais e os valores obtidos após aplicação de `trim()` e `lower()` demonstrou que não existem diferenças decorrentes de espaços adicionais ou uso inconsistente de letras maiúsculas e minúsculas.

Apesar da consistência observada na camada Bronze, a camada Silver aplicará a padronização dos campos textuais como medida preventiva, garantindo uniformidade para as etapas posteriores do pipeline.

## 5. Consistência dos valores numéricos

Nesta etapa são avaliados os principais atributos quantitativos do conjunto de dados, buscando identificar valores negativos, valores iguais a zero, intervalos potencialmente inconsistentes e possíveis valores extremos antes da construção da camada Silver.

In [0]:
from pyspark.sql.functions import col, min, max, avg

campos_numericos = [
    "mediana_m2",
    "p25_m2",
    "p75_m2",
    "anuncios_ativos_media_dia",
    "novos_anuncios",
    "dias_no_mercado_medio"
]

# Estatísticas básicas
df_bronze.select(
    *[
        funcao(col(campo)).alias(f"{nome}_{campo}")
        for campo in campos_numericos
        for funcao, nome in [(min, "min"), (max, "max"), (avg, "media")]
    ]
).show(truncate=False)

# Contagem de valores negativos e iguais a zero
for campo in campos_numericos:
    negativos = df_bronze.filter(col(campo) < 0).count()
    zeros = df_bronze.filter(col(campo) == 0).count()

    print(
        f"{campo}: "
        f"negativos = {negativos} | "
        f"zeros = {zeros}"
    )

+--------------+--------------+-----------------+----------+----------+-----------------+----------+----------+----------------+-----------------------------+-----------------------------+-------------------------------+------------------+------------------+--------------------+-------------------------+-------------------------+---------------------------+
|min_mediana_m2|max_mediana_m2|media_mediana_m2 |min_p25_m2|max_p25_m2|media_p25_m2     |min_p75_m2|max_p75_m2|media_p75_m2    |min_anuncios_ativos_media_dia|max_anuncios_ativos_media_dia|media_anuncios_ativos_media_dia|min_novos_anuncios|max_novos_anuncios|media_novos_anuncios|min_dias_no_mercado_medio|max_dias_no_mercado_medio|media_dias_no_mercado_medio|
+--------------+--------------+-----------------+----------+----------+-----------------+----------+----------+----------------+-----------------------------+-----------------------------+-------------------------------+------------------+------------------+--------------------+-

### 5.1 Investigação de valores potencialmente inconsistentes

A análise inicial não identificou valores negativos nos atributos numéricos. Entretanto, foram observados valores iguais a zero nas métricas de preço por metro quadrado e valores máximos muito superiores às respectivas médias.

Antes da aplicação de qualquer tratamento, esses registros serão investigados individualmente para determinar sua distribuição, origem e possível impacto nas análises posteriores.

In [0]:
from pyspark.sql.functions import col

# Registros em que alguma métrica de preço é igual a zero
df_precos_zero = (
    df_bronze
    .filter(
        (col("mediana_m2") == 0) |
        (col("p25_m2") == 0) |
        (col("p75_m2") == 0)
    )
    .select(
        "mes",
        "cidade",
        "uf",
        "bairro",
        "transacao",
        "mediana_m2",
        "p25_m2",
        "p75_m2",
        "anuncios_ativos_media_dia",
        "novos_anuncios",
        "dias_no_mercado_medio"
    )
)

print("REGISTROS COM PREÇO/M² IGUAL A ZERO:")
display(df_precos_zero.orderBy("cidade", "bairro", "transacao"))

# Maiores valores observados de mediana_m2
print("MAIORES VALORES DE MEDIANA_M2:")
display(
    df_bronze
    .select(
        "mes", "cidade", "uf", "bairro", "transacao",
        "mediana_m2", "p25_m2", "p75_m2",
        "anuncios_ativos_media_dia",
        "novos_anuncios",
        "dias_no_mercado_medio"
    )
    .orderBy(col("mediana_m2").desc())
    .limit(20)
)

REGISTROS COM PREÇO/M² IGUAL A ZERO:


mes,cidade,uf,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio
2026-06-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,289
2026-02-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,168
2026-03-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,197
2026-05-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,258
2025-10-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,46
2026-01-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,138
2025-09-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,16
2025-12-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,107
2025-11-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,77
2026-04-01,rio-de-janeiro,rj,Camorim,aluguel,0,0,0,1,0,228


MAIORES VALORES DE MEDIANA_M2:


mes,cidade,uf,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio
2026-07-01,sao-paulo,sp,Jardim Itatinga,venda,3575429,7056,7669643,5,0,548
2026-06-01,sao-paulo,sp,Jardim Itatinga,venda,2930087,7404,6283460,6,0,437
2026-05-01,recife,pe,Tejipió,venda,2381041,1191562,3570521,3,0,102
2026-04-01,recife,pe,Tejipió,venda,2381041,1191562,3570521,2,1,93
2026-03-01,recife,pe,Tejipió,venda,2381041,1191562,3570521,2,0,79
2026-05-01,sao-paulo,sp,Jardim Itatinga,venda,2076680,7305,4450979,6,3,413
2026-06-01,recife,pe,Tejipió,venda,1112386,557234,2936193,3,2,115
2025-11-01,sao-paulo,sp,Vila Nova Galvão,venda,1063810,1063810,1063810,1,1,1
2026-08-01,sao-paulo,sp,Não Informado,venda,679874,449329,2854659,90,0,248
2026-09-01,sao-paulo,sp,Não Informado,venda,679874,449329,2854659,90,0,272


### 5.1.1 Resultado da investigação dos valores numéricos

A investigação demonstrou que os **11 registros com valor igual a zero nas métricas de preço por metro quadrado** pertencem à mesma combinação de mercado: **Camorim, Rio de Janeiro/RJ, transação de aluguel**. Esses registros apresentam apenas um anúncio ativo e não possuem novos anúncios no período.

Como preço por metro quadrado igual a zero não representa um valor economicamente válido para uma oferta imobiliária, esses valores serão considerados ausência de informação e convertidos para `NULL` na camada Silver, preservando-se os demais atributos dos registros.

Também foram identificados valores de preço por metro quadrado muito superiores às médias do conjunto. Esses registros não serão removidos automaticamente, pois valores elevados podem decorrer de características específicas dos dados ou de segmentos com baixa quantidade de anúncios. Antes de qualquer tratamento, será realizada uma análise estatística de outliers.

### 5.2 Análise estatística de outliers pelo IQR

In [0]:
from pyspark.sql.functions import col

campos_preco = ["mediana_m2", "p25_m2", "p75_m2"]

for campo in campos_preco:

    # Desconsidera nulos e zeros nesta análise
    df_valido = df_bronze.filter(
        col(campo).isNotNull() & (col(campo) > 0)
    )

    q1, q3 = df_valido.approxQuantile(campo, [0.25, 0.75], 0.01)

    iqr = q3 - q1
    limite_inferior = q1 - (1.5 * iqr)
    limite_superior = q3 + (1.5 * iqr)

    quantidade_outliers = (
        df_valido
        .filter(
            (col(campo) < limite_inferior) |
            (col(campo) > limite_superior)
        )
        .count()
    )

    print(f"\nCAMPO: {campo}")
    print(f"Q1: {q1}")
    print(f"Q3: {q3}")
    print(f"IQR: {iqr}")
    print(f"Limite inferior: {limite_inferior}")
    print(f"Limite superior: {limite_superior}")
    print(f"Quantidade de potenciais outliers: {quantidade_outliers}")


CAMPO: mediana_m2
Q1: 53.0
Q3: 6290.0
IQR: 6237.0
Limite inferior: -9302.5
Limite superior: 15645.5
Quantidade de potenciais outliers: 803

CAMPO: p25_m2
Q1: 47.0
Q3: 5621.0
IQR: 5574.0
Limite inferior: -8314.0
Limite superior: 13982.0
Quantidade de potenciais outliers: 619

CAMPO: p75_m2
Q1: 58.0
Q3: 6832.0
IQR: 6774.0
Limite inferior: -10103.0
Limite superior: 16993.0
Quantidade de potenciais outliers: 996


### 5.2.1 Análise dos potenciais outliers

A aplicação do método do intervalo interquartil (IQR) identificou valores potencialmente extremos nas três métricas de preço analisadas.

Para `mediana_m2`, foram identificados **803 potenciais outliers**, considerando limite superior de **15.645,50**. Para `p25_m2`, foram encontrados **619 registros** acima do limite superior de **13.982,00**, enquanto `p75_m2` apresentou **996 registros** acima do limite de **16.993,00**.

Entretanto, esses registros não serão removidos automaticamente. A base reúne mercados distintos, incluindo operações de **venda e aluguel**, diferentes cidades e diversos bairros, o que produz distribuições de preços naturalmente heterogêneas. Além disso, alguns valores elevados estão associados a grupos com pequena quantidade de anúncios, podendo produzir métricas mais sensíveis a observações individuais.

Dessa forma, o método IQR foi utilizado como instrumento de diagnóstico e identificação de valores potencialmente atípicos, e não como critério automático de exclusão. A preservação desses registros evita a eliminação indevida de valores que podem representar características reais dos mercados analisados.

## 6. Construção da camada Silver

Após a avaliação da qualidade dos dados da camada Bronze, inicia-se a etapa de transformação e padronização.

Na camada Silver serão aplicadas as seguintes regras:

- padronização dos campos textuais;
- conversão dos valores iguais a zero nas métricas de preço para `NULL`;
- preservação dos valores ausentes originalmente existentes;
- manutenção dos valores elevados identificados pelo método IQR, uma vez que não há evidência suficiente para classificá-los automaticamente como erros;
- preservação dos registros, visto que não foram encontradas duplicidades completas;
- manutenção dos tipos de dados adequados para as análises posteriores.

O objetivo é produzir uma base limpa e consistente, mantendo a rastreabilidade das decisões de tratamento adotadas.

In [0]:
from pyspark.sql.functions import (
    col, when, trim, lower
)

df_silver = (
    df_bronze

    # Padronização dos campos textuais
    .withColumn("bairro", trim(col("bairro")))
    .withColumn("cidade", lower(trim(col("cidade"))))
    .withColumn("uf", lower(trim(col("uf"))))
    .withColumn("transacao", lower(trim(col("transacao"))))

    # Valores zero nas métricas de preço passam a representar ausência
    .withColumn(
        "mediana_m2",
        when(col("mediana_m2") == 0, None)
        .otherwise(col("mediana_m2"))
    )

    .withColumn(
        "p25_m2",
        when(col("p25_m2") == 0, None)
        .otherwise(col("p25_m2"))
    )

    .withColumn(
        "p75_m2",
        when(col("p75_m2") == 0, None)
        .otherwise(col("p75_m2"))
    )
)

print(f"Registros na Bronze: {df_bronze.count()}")
print(f"Registros após as transformações Silver: {df_silver.count()}")

display(df_silver.limit(10))

Registros na Bronze: 36933
Registros após as transformações Silver: 36933


mes,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio,arquivo_origem,cidade,uf,data_ingestao
2025-09-01,Abranches,venda,5941,4341,6781,42,1,511,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Água Verde,aluguel,50,41,59,4,0,286,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Água Verde,venda,11771,9692,14377,533,11,384,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Ahú,aluguel,48,48,48,2,0,587,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Ahú,venda,12235,11027,13344,126,1,352,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto Boqueirão,aluguel,25,24,25,3,0,144,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto Boqueirão,venda,4866,4375,5513,58,1,366,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Glória,aluguel,72,52,86,14,0,784,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Glória,venda,12762,10920,15346,86,1,443,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Rua XV,aluguel,26,26,26,1,0,104,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z


### 6.1 Validação das transformações da camada Silver

Antes da persistência definitiva da camada Silver, são realizadas verificações para confirmar se as regras de tratamento foram aplicadas corretamente.

Nesta etapa serão comparadas as quantidades de valores nulos e iguais a zero nas principais métricas de preço, permitindo verificar o efeito das transformações realizadas sem alteração da quantidade total de registros.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

campos_preco = ["mediana_m2", "p25_m2", "p75_m2"]

for campo in campos_preco:
    nulos_bronze = df_bronze.filter(col(campo).isNull()).count()
    zeros_bronze = df_bronze.filter(col(campo) == 0).count()

    nulos_silver = df_silver.filter(col(campo).isNull()).count()
    zeros_silver = df_silver.filter(col(campo) == 0).count()

    print(f"CAMPO: {campo}")
    print(f"Bronze -> nulos: {nulos_bronze} | zeros: {zeros_bronze}")
    print(f"Silver -> nulos: {nulos_silver} | zeros: {zeros_silver}")
    print("-" * 50)

CAMPO: mediana_m2
Bronze -> nulos: 3756 | zeros: 11
Silver -> nulos: 3767 | zeros: 0
--------------------------------------------------
CAMPO: p25_m2
Bronze -> nulos: 3756 | zeros: 11
Silver -> nulos: 3767 | zeros: 0
--------------------------------------------------
CAMPO: p75_m2
Bronze -> nulos: 3756 | zeros: 11
Silver -> nulos: 3767 | zeros: 0
--------------------------------------------------


### 6.1.1 Resultado da validação dos valores de preço

A validação confirmou que a transformação foi aplicada corretamente às três métricas de preço (`mediana_m2`, `p25_m2` e `p75_m2`).

Na camada Bronze, cada uma dessas variáveis apresentava **3.756 valores nulos e 11 registros com valor igual a zero**. Após a transformação, a camada Silver passou a apresentar **3.767 valores nulos e nenhum registro com valor igual a zero**.

Portanto, os 11 valores iguais a zero foram corretamente convertidos para `NULL`, preservando os valores originalmente ausentes e mantendo a quantidade total de **36.933 registros**.

Esse tratamento evita que a ausência de informação de preço seja interpretada como um preço real igual a zero, o que poderia distorcer as análises estatísticas posteriores.

### 6.2 Validação dos campos categóricos 

In [0]:
# Validação dos campos categóricos após a transformação

print("CIDADES:")
df_silver.select("cidade").distinct().orderBy("cidade").show(100, truncate=False)

print("UNIDADES FEDERATIVAS:")
df_silver.select("uf").distinct().orderBy("uf").show(100, truncate=False)

print("TIPOS DE TRANSAÇÃO:")
df_silver.select("transacao").distinct().orderBy("transacao").show(100, truncate=False)

CIDADES:
+--------------+
|cidade        |
+--------------+
|curitiba      |
|recife        |
|rio-de-janeiro|
|salvador      |
|sao-paulo     |
+--------------+

UNIDADES FEDERATIVAS:
+---+
|uf |
+---+
|ba |
|pe |
|pr |
|rj |
|sp |
+---+

TIPOS DE TRANSAÇÃO:
+---------+
|transacao|
+---------+
|aluguel  |
|venda    |
+---------+



## 6.3 Persistência da camada Silver

Após a execução e validação das regras de tratamento, os dados transformados são persistidos na camada Silver em formato Delta.

A tabela Silver contém os dados padronizados e preparados para as etapas posteriores de modelagem e análise, mantendo os **36.933 registros** provenientes da camada Bronze.

As principais transformações realizadas foram a padronização dos campos categóricos e o tratamento dos valores iguais a zero nas métricas de preço, convertidos para valores nulos quando representavam ausência de informação.

In [0]:
# Persistência da camada Silver no Unity Catalog

tabela_silver = "mvp_imobiliario.silver.mercado_imobiliario_tratado"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)

print(f"Tabela Silver criada com sucesso: {tabela_silver}")

Tabela Silver criada com sucesso: mvp_imobiliario.silver.mercado_imobiliario_tratado


### 6.3.1 Validação da tabela Silver persistida

Após a persistência, a tabela Silver é novamente consultada diretamente no Unity Catalog. Essa verificação permite confirmar que os dados foram efetivamente armazenados no ambiente de nuvem e que a quantidade de registros foi preservada após as transformações.

In [0]:
# Validação da tabela Silver persistida no Unity Catalog

df_silver_gravado = spark.table(
    "mvp_imobiliario.silver.mercado_imobiliario_tratado"
)

print(
    f"Quantidade de registros gravados na Silver: "
    f"{df_silver_gravado.count()}"
)

display(df_silver_gravado.limit(10))

Quantidade de registros gravados na Silver: 36933


mes,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio,arquivo_origem,cidade,uf,data_ingestao
2025-09-01,Abranches,venda,5941,4341,6781,42,1,511,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Água Verde,aluguel,50,41,59,4,0,286,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Água Verde,venda,11771,9692,14377,533,11,384,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Ahú,aluguel,48,48,48,2,0,587,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Ahú,venda,12235,11027,13344,126,1,352,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto Boqueirão,aluguel,25,24,25,3,0,144,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto Boqueirão,venda,4866,4375,5513,58,1,366,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Glória,aluguel,72,52,86,14,0,784,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Glória,venda,12762,10920,15346,86,1,443,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z
2025-09-01,Alto da Rua XV,aluguel,26,26,26,1,0,104,dbfs:/Volumes/mvp_imobiliario/bronze/dados_origem/guru-dados-curitiba-pr.csv,curitiba,pr,2026-09-22T01:33:53.175Z


### 6.4 Documentação da camada Silver no Unity Catalog

A tabela da camada Silver é documentada no Unity Catalog com uma descrição de sua finalidade e das principais transformações realizadas. Também são adicionadas descrições às colunas, contribuindo para a construção do catálogo de dados e para a compreensão da estrutura utilizada no pipeline.

In [0]:
# Documentação da tabela Silver no Unity Catalog

spark.sql("""
COMMENT ON TABLE mvp_imobiliario.silver.mercado_imobiliario_tratado IS
'Tabela tratada do mercado imobiliário utilizada na camada Silver. Contém dados de cinco cidades brasileiras após padronização dos campos categóricos e tratamento das métricas de preço.'
""")

comentarios_colunas = {
    "mes": "Mês de referência das métricas do mercado imobiliário.",
    "bairro": "Nome do bairro ao qual as métricas se referem.",
    "transacao": "Tipo de transação imobiliária: venda ou aluguel.",
    "mediana_m2": "Valor mediano do preço por metro quadrado.",
    "p25_m2": "Percentil 25 do preço por metro quadrado.",
    "p75_m2": "Percentil 75 do preço por metro quadrado.",
    "anuncios_ativos_media_dia": "Quantidade média diária de anúncios ativos.",
    "novos_anuncios": "Quantidade de novos anúncios no período.",
    "dias_no_mercado_medio": "Quantidade média de dias dos imóveis no mercado.",
    "arquivo_origem": "Caminho do arquivo CSV utilizado como fonte do registro.",
    "cidade": "Cidade correspondente ao mercado imobiliário analisado.",
    "uf": "Unidade Federativa correspondente à cidade.",
    "data_ingestao": "Data e hora em que o registro foi ingerido na camada Bronze."
}

for coluna, descricao in comentarios_colunas.items():
    spark.sql(f"""
        ALTER TABLE mvp_imobiliario.silver.mercado_imobiliario_tratado
        ALTER COLUMN {coluna}
        COMMENT '{descricao}'
    """)

print("Documentação da tabela Silver concluída com sucesso.")

Documentação da tabela Silver concluída com sucesso.
